This notebook provides examples to go along with the [textbook](http://manipulation.csail.mit.edu/clutter.html).  I recommend having both windows open, side-by-side!

In [ ]:
import numpy as np
from IPython.display import clear_output
from pydrake.all import (
    AbstractValue,
    AddMultibodyPlantSceneGraph,
    Box,
    Capsule,
    ContactResults,
    ContactVisualizer,
    ContactVisualizerParams,
    Cylinder,
    DiagramBuilder,
    Ellipsoid,
    FixedOffsetFrame,
    JointSliders,
    LeafSystem,
    MeshcatVisualizer,
    Parser,
    PlanarJoint,
    RigidTransform,
    RotationMatrix,
    Sphere,
    StartMeshcat,
)

from manipulation import running_as_notebook
from manipulation.scenarios import AddShape
from manipulation.utils import ConfigureParser

In [ ]:
# Start the visualizer.
meshcat = StartMeshcat()

# Contact force "inspector"


In [ ]:
class PrintContactResults(LeafSystem):
    def __init__(self):
        LeafSystem.__init__(self)
        self.DeclareAbstractInputPort(
            "contact_results", AbstractValue.Make(ContactResults())
        )
        self.DeclareForcedPublishEvent(self.Publish)
        self.DeclarePeriodicPublishEvent(0.1, 0, self.Publish)

    def Publish(self, context):
        formatter = {"float": lambda x: "{:5.2f}".format(x)}
        results = self.get_input_port().Eval(context)

        if results.num_hydroelastic_contacts() == 0:
            print("no contact")
        for i in range(results.num_hydroelastic_contacts()):
            info = results.hydroelastic_contact_info(i)
            print(f"F_Ac_W = {info.F_Ac_W()}")

        clear_output(wait=True)


def contact_force_inspector(slope=0.0, mu=1.0, second_brick=False):
    builder = DiagramBuilder()

    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.01)

    AddShape(
        plant,
        Box(10.0, 10.0, 10.0),
        "ground",
        mass=1,
        mu=1,
        color=[0.9, 0.9, 0.9, 0.8],
    )
    X_WBox = RigidTransform(RotationMatrix.MakeYRotation(slope), [0, 0, -5.05])
    plant.WeldFrames(plant.world_frame(), plant.GetFrameByName("ground"), X_WBox)

    parser = Parser(plant)
    ConfigureParser(parser)
    parser.AddModelsFromUrl("package://manipulation/hydro/061_foam_brick.sdf")
    frame = plant.AddFrame(
        FixedOffsetFrame(
            "planar_joint_frame",
            plant.world_frame(),
            RigidTransform(RotationMatrix.MakeXRotation(np.pi / 2)),
        )
    )
    plant.AddJoint(
        PlanarJoint(
            "brick",
            frame,
            plant.GetFrameByName("base_link"),
            damping=[0, 0, 0],
        )
    )

    if second_brick:
        parser.SetAutoRenaming(True)
        brick2 = parser.AddModelsFromUrl(
            "package://manipulation/hydro/061_foam_brick.sdf"
        )[0]
        plant.AddJoint(
            PlanarJoint(
                "brick2",
                frame,
                plant.GetFrameByName("base_link", brick2),
                damping=[0, 0, 0],
            )
        )

    plant.Finalize()

    meshcat.Delete()
    meshcat.DeleteAddedControls()
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)
    meshcat.Set2dRenderMode(xmin=-0.2, xmax=0.2, ymin=-0.2, ymax=0.3)

    cparams = ContactVisualizerParams()
    cparams.force_threshold = 1e-6
    cparams.newtons_per_meter = 1.0
    cparams.radius = 0.002
    contact_visualizer = ContactVisualizer.AddToBuilder(
        builder, plant, meshcat, cparams
    )

    print_contact_results = builder.AddSystem(PrintContactResults())
    builder.Connect(
        plant.get_contact_results_output_port(),
        print_contact_results.get_input_port(),
    )

    lower_limit = [-0.1, -0.1, -np.pi / 2.0]
    upper_limit = [0.1, 0.1, np.pi / 2.0]
    q0 = [0, 0, 0]
    if second_brick:
        lower_limit += lower_limit
        upper_limit += upper_limit
        q0 += [0.07, 0.07, 0.0]

    default_interactive_timeout = None if running_as_notebook else 1.0
    sliders = builder.AddSystem(
        JointSliders(
            meshcat,
            plant,
            initial_value=q0,
            lower_limit=lower_limit,
            upper_limit=upper_limit,
            step=0.001,
        )
    )
    diagram = builder.Build()
    sliders.Run(diagram, default_interactive_timeout)
    meshcat.DeleteAddedControls()


contact_force_inspector(slope=0.1, mu=0.5, second_brick=True)

# Contact results "inspector"

This simple visualization shows some of the complexity of the contact geometry problem.  I will make it better, but right now, when you move the objects into contact of each other you will see three points:  the contact point is in **red**, the contact normal is added to the contact point with the tip as **green**, and the (scaled) contact force tip is drawn in **blue**.  Contact points on the bodies are drawn in **orange**.

In [ ]:
shapes = {
    "Point": Sphere(0.01),
    "Sphere": Sphere(1.0),
    "Cylinder": Cylinder(1.0, 2.0),
    "Box": Box(1.0, 2.0, 3.0),
    "Capsule": Capsule(1.0, 2.0),
    "Ellipsoid": Ellipsoid(1.0, 2.0, 3.0),
}


def contact_inspector(shape_name_A, shape_name_B):
    builder = DiagramBuilder()

    plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
    AddShape(
        plant,
        shapes[shape_name_A],
        "A",
        mass=1,
        mu=1,
        color=[0.9, 0.5, 0.5, 0.5],
    )
    plant.WeldFrames(plant.world_frame(), plant.GetFrameByName("A"))

    AddShape(
        plant,
        shapes[shape_name_B],
        "B",
        mass=1,
        mu=1,
        color=[0.5, 0.5, 0.9, 0.5],
    )
    frame = plant.AddFrame(
        FixedOffsetFrame(
            "planar_joint_frame",
            plant.world_frame(),
            RigidTransform(RotationMatrix.MakeXRotation(np.pi / 2)),
        )
    )
    plant.AddJoint(
        PlanarJoint("B", frame, plant.GetFrameByName("B"), damping=[0, 0, 0])
    )

    plant.Finalize()

    meshcat.Delete()
    meshcat.DeleteAddedControls()
    MeshcatVisualizer.AddToBuilder(builder, scene_graph, meshcat)
    meshcat.Set2dRenderMode(xmin=-3.0, xmax=3.0, ymin=-0.2, ymax=3.0)

    cparams = ContactVisualizerParams()
    cparams.force_threshold = 1e-6
    cparams.newtons_per_meter = 1.0
    cparams.radius = 0.002
    contact_visualizer = ContactVisualizer.AddToBuilder(
        builder, plant, meshcat, cparams
    )

    print_contact_results = builder.AddSystem(PrintContactResults())
    builder.Connect(
        plant.get_contact_results_output_port(),
        print_contact_results.get_input_port(),
    )

    lower_limit = [-3, -3, -np.pi / 2.0]
    upper_limit = [3, 3, np.pi / 2.0]
    q0 = [1.2, 1.2, 0.0]

    default_interactive_timeout = None if running_as_notebook else 1.0
    sliders = builder.AddSystem(
        JointSliders(
            meshcat,
            plant,
            initial_value=q0,
            lower_limit=lower_limit,
            upper_limit=upper_limit,
        )
    )
    diagram = builder.Build()
    sliders.Run(diagram, default_interactive_timeout)
    meshcat.DeleteAddedControls()


contact_inspector("Box", "Sphere")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=8e10b301-2776-448f-be43-c7f6fb54fa1f' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>